# cyp51A Gene Variants Analysis - Iteration 2

**Fixed ModuleNotFoundError**: Using only standard libraries available in JupyterLite

This notebook identifies and visualizes variants falling within the cyp51A gene (Afu4g06890) using:
- GTF dataset #4 for gene structure
- VCF file collections from history #450 and #351 for variant data
- Creates heatmap showing variants in coding regions with local coordinates

In [ ]:
# Import only standard libraries available in JupyterLite
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle
import matplotlib.colors as mcolors
import re
import os
import glob
import warnings
warnings.filterwarnings('ignore')

# Set up plotting parameters
plt.style.use('default')
plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['font.size'] = 10

print("✓ Successfully imported all required libraries")
print("Available libraries: pandas, matplotlib, numpy, re, os")

## Step 1: Load GTF Data and Find cyp51A Gene

In [ ]:
# Load GTF file (dataset #4)
print("Loading GTF data from dataset #4...")

# Try different possible filenames for dataset #4
possible_gtf_files = ['dataset_4.dat', 'dataset_4.gtf', 'dataset_4.txt']
gtf_file = None

for filename in possible_gtf_files:
    if os.path.exists(filename):
        gtf_file = filename
        print(f"✓ Found GTF file: {filename}")
        break

if gtf_file is None:
    print("⚠️ GTF file not found. Available files:")
    available_files = [f for f in os.listdir('.') if f.startswith('dataset') and any(x in f for x in ['4', 'gtf'])]
    print(available_files[:10])
    # Use the first available dataset file as fallback
    if available_files:
        gtf_file = available_files[0]
        print(f"Using fallback file: {gtf_file}")

gtf_columns = ['seqname', 'source', 'feature', 'start', 'end', 'score', 'strand', 'frame', 'attribute']

if gtf_file and os.path.exists(gtf_file):
    try:
        gtf_data = pd.read_csv(gtf_file, sep='\t', comment='#', names=gtf_columns, dtype=str)
        print(f"✓ Loaded GTF file with {len(gtf_data)} entries")
        print(f"Columns: {gtf_data.columns.tolist()}")
    except Exception as e:
        print(f"✗ Error loading GTF file: {e}")
        gtf_data = None
else:
    print("✗ No GTF file available")
    gtf_data = None

In [ ]:
# Find cyp51A gene (Afu4g06890)
def extract_gene_info(attribute_string):
    """Extract gene ID and other info from GTF attributes"""
    if pd.isna(attribute_string):
        return None, None
    
    gene_id = None
    gene_name = None
    
    # Look for gene_id, ID, Name, locus_tag
    patterns = [
        (r'gene_id\s*["=]([^\s;";]+)', 'gene_id'),
        (r'ID=([^\s;]+)', 'ID'),
        (r'Name=([^\s;]+)', 'Name'),
        (r'locus_tag\s*["=]([^\s;";]+)', 'locus_tag')
    ]
    
    for pattern, field in patterns:
        match = re.search(pattern, str(attribute_string), re.IGNORECASE)
        if match:
            if field in ['gene_id', 'ID', 'locus_tag']:
                gene_id = match.group(1)
            elif field == 'Name':
                gene_name = match.group(1)
    
    return gene_id, gene_name

if gtf_data is not None:
    print("Searching for cyp51A gene (Afu4g06890)...")
    
    # Extract gene info
    gene_info = gtf_data['attribute'].apply(extract_gene_info)
    gtf_data['gene_id'] = [info[0] for info in gene_info]
    gtf_data['gene_name'] = [info[1] for info in gene_info]
    
    # Search for cyp51A gene (Afu4g06890)
    cyp51a_mask = (
        gtf_data['attribute'].str.contains('Afu4g06890', case=False, na=False) |
        gtf_data['gene_id'].str.contains('Afu4g06890', case=False, na=False) |
        gtf_data['attribute'].str.contains('cyp51A', case=False, na=False)
    )
    
    cyp51a_entries = gtf_data[cyp51a_mask].copy()
    
    print(f"Found {len(cyp51a_entries)} entries for cyp51A (Afu4g06890)")
    
    if len(cyp51a_entries) > 0:
        print("\ncyp51A gene entries:")
        display_cols = ['seqname', 'feature', 'start', 'end', 'strand']
        print(cyp51a_entries[display_cols].head(10).to_string())
        
        # Get basic gene info
        chromosome = cyp51a_entries.iloc[0]['seqname']
        strand = cyp51a_entries.iloc[0]['strand']
        
        # Convert coordinates to int
        cyp51a_entries['start'] = cyp51a_entries['start'].astype(int)
        cyp51a_entries['end'] = cyp51a_entries['end'].astype(int)
        
        gene_start = cyp51a_entries['start'].min()
        gene_end = cyp51a_entries['end'].max()
        
        print(f"\ncyp51A gene location: {chromosome}:{gene_start:,}-{gene_end:,} ({strand} strand)")
        gene_found = True
        
    else:
        print("\n⚠️ cyp51A gene not found. Searching for similar entries...")
        similar = gtf_data[gtf_data['attribute'].str.contains('cyp|CYP', case=False, na=False)]
        print(f"Found {len(similar)} entries containing 'cyp'")
        if len(similar) > 0:
            print(similar[['seqname', 'feature', 'attribute']].head().to_string())
        gene_found = False
else:
    print("GTF data not available")
    gene_found = False

## Step 2: Extract CDS Regions and Create Local Coordinate Mapping

In [ ]:
if gene_found and len(cyp51a_entries) > 0:
    print("Extracting CDS regions and creating coordinate mapping...")
    
    # Extract CDS entries
    cds_entries = cyp51a_entries[cyp51a_entries['feature'] == 'CDS'].copy()
    
    if len(cds_entries) == 0:
        # Try exon or other features if no CDS
        cds_entries = cyp51a_entries[cyp51a_entries['feature'].isin(['exon', 'mRNA', 'transcript'])].copy()
        print(f"No CDS entries found, using {len(cds_entries)} {cds_entries['feature'].iloc[0] if len(cds_entries) > 0 else 'N/A'} entries")
    else:
        print(f"Found {len(cds_entries)} CDS entries")
    
    if len(cds_entries) > 0:
        # Get CDS coordinates
        cds_coords = []
        for _, cds in cds_entries.iterrows():
            start, end = int(cds['start']), int(cds['end'])
            cds_coords.append((start, end))
            print(f"  CDS: {start:,} - {end:,} ({end-start+1} bp)")
        
        # Sort CDS coordinates
        cds_coords.sort()
        
        # Create local coordinate mapping (1-based, coding sequence only)
        local_coords = {}  # genomic_pos -> local_pos
        genomic_to_local = {}  # local_pos -> genomic_pos
        local_pos = 1
        
        print(f"\nCreating local coordinate mapping:")
        for i, (start, end) in enumerate(cds_coords, 1):
            cds_length = end - start + 1
            print(f"  Exon {i}: genomic {start:,}-{end:,} → local {local_pos}-{local_pos + cds_length - 1}")
            
            for genomic_pos in range(start, end + 1):
                local_coords[genomic_pos] = local_pos
                genomic_to_local[local_pos] = genomic_pos
                local_pos += 1
        
        total_coding_length = local_pos - 1
        print(f"\nTotal coding sequence: {total_coding_length} nucleotides")
        
        # Define region of interest for variant analysis
        roi_start = gene_start - 1000  # 1kb upstream
        roi_end = gene_end + 1000      # 1kb downstream
        
        print(f"Region of interest: {chromosome}:{roi_start:,}-{roi_end:,}")
        coordinates_available = True
        
    else:
        print("No CDS or exon entries found for cyp51A")
        cds_coords = []
        local_coords = {}
        total_coding_length = 0
        coordinates_available = False

else:
    print("cyp51A gene data not available")
    cds_coords = []
    local_coords = {}
    total_coding_length = 0
    coordinates_available = False

## Step 3: Load VCF Files from Collections #450 and #351

In [ ]:
# Search for VCF files from history collections #450 and #351
print("Searching for VCF files from collections #450 and #351...")

# List all available files
all_files = os.listdir('.')
print(f"Total files in directory: {len(all_files)}")

# Look for files that might be VCF from collections 450 and 351
target_patterns = ['450', '351', 'vcf', 'VCF']
potential_vcf_files = []

for f in all_files:
    if any(pattern in f for pattern in target_patterns):
        potential_vcf_files.append(f)

print(f"\nPotential VCF files found: {len(potential_vcf_files)}")
for f in potential_vcf_files[:10]:  # Show first 10
    print(f"  {f}")

# Function to check if a file is VCF format
def is_vcf_file(filename):
    """Check if file appears to be VCF format"""
    try:
        with open(filename, 'r') as f:
            lines = []
            for i, line in enumerate(f):
                lines.append(line.strip())
                if i >= 20:  # Check first 20 lines
                    break
        
        # Check for VCF indicators
        has_vcf_header = any('##fileformat=VCF' in line for line in lines)
        has_chrom_line = any(line.startswith('#CHROM') for line in lines)
        has_vcf_content = any(len(line.split('\t')) >= 8 for line in lines if not line.startswith('#'))
        
        return has_vcf_header or (has_chrom_line and has_vcf_content)
        
    except Exception as e:
        return False

# Check which files are actually VCF format
vcf_files = []
for filename in potential_vcf_files:
    if os.path.exists(filename) and is_vcf_file(filename):
        vcf_files.append(filename)
        print(f"✓ Confirmed VCF file: {filename}")

print(f"\nConfirmed VCF files: {len(vcf_files)}")

In [ ]:
# Load and process VCF files
vcf_data_loaded = {}
all_variants = []

if len(vcf_files) > 0:
    print("Loading VCF data...")
    
    for vcf_file in vcf_files[:3]:  # Limit to first 3 files to avoid memory issues
        print(f"\nProcessing {vcf_file}...")
        
        try:
            # Read VCF file
            with open(vcf_file, 'r') as f:
                lines = f.readlines()
            
            # Find header line
            header_idx = -1
            for i, line in enumerate(lines):
                if line.startswith('#CHROM'):
                    header_idx = i
                    break
            
            if header_idx >= 0:
                # Get column names from header
                header_line = lines[header_idx].strip().replace('#', '')
                col_names = header_line.split('\t')
                
                # Read data lines
                data_lines = []
                for line in lines[header_idx + 1:]:
                    if not line.startswith('#') and line.strip():
                        data_lines.append(line.strip().split('\t'))
                
                if len(data_lines) > 0:
                    # Create DataFrame
                    # Pad shorter rows to match header length
                    max_cols = len(col_names)
                    padded_data = []
                    for row in data_lines:
                        if len(row) < max_cols:
                            row.extend([''] * (max_cols - len(row)))
                        elif len(row) > max_cols:
                            row = row[:max_cols]
                        padded_data.append(row)
                    
                    vcf_df = pd.DataFrame(padded_data, columns=col_names)
                    
                    # Convert POS to numeric
                    vcf_df['POS'] = pd.to_numeric(vcf_df['POS'], errors='coerce')
                    
                    # Filter for cyp51A region if coordinates available
                    if coordinates_available and 'chromosome' in locals():
                        region_variants = vcf_df[
                            (vcf_df['CHROM'] == chromosome) &
                            (vcf_df['POS'] >= roi_start) &
                            (vcf_df['POS'] <= roi_end)
                        ].copy()
                        
                        if len(region_variants) > 0:
                            region_variants['source_file'] = vcf_file
                            vcf_data_loaded[vcf_file] = region_variants
                            all_variants.append(region_variants)
                            print(f"  ✓ Found {len(region_variants)} variants in cyp51A region")
                        else:
                            print(f"  No variants in cyp51A region")
                    else:
                        # No coordinates available, take first 100 variants
                        sample_variants = vcf_df.head(100).copy()
                        sample_variants['source_file'] = vcf_file
                        vcf_data_loaded[vcf_file] = sample_variants
                        all_variants.append(sample_variants)
                        print(f"  ✓ Loaded {len(sample_variants)} sample variants")
                else:
                    print(f"  No data lines found")
            else:
                print(f"  No header line found")
                
        except Exception as e:
            print(f"  ✗ Error processing {vcf_file}: {str(e)[:100]}...")
    
    print(f"\nTotal VCF files loaded: {len(vcf_data_loaded)}")
else:
    print("No VCF files found")

# Combine all variant data
if len(all_variants) > 0:
    combined_variants = pd.concat(all_variants, ignore_index=True)
    print(f"Combined variants dataset: {len(combined_variants)} total variants")
else:
    combined_variants = pd.DataFrame()
    print("No variant data available")

## Step 4: Analyze Variants in Coding Regions

In [ ]:
# Analyze variants that fall in coding regions
coding_variants = pd.DataFrame()
variant_summary = pd.DataFrame()

if len(combined_variants) > 0 and coordinates_available:
    print("Analyzing variants in coding regions...")
    
    # Filter variants that fall in coding regions
    combined_variants['POS'] = pd.to_numeric(combined_variants['POS'], errors='coerce')
    coding_mask = combined_variants['POS'].isin(local_coords.keys())
    coding_variants = combined_variants[coding_mask].copy()
    
    if len(coding_variants) > 0:
        # Add local coordinates
        coding_variants['local_pos'] = coding_variants['POS'].map(local_coords)
        
        print(f"\nFound {len(coding_variants)} variants in coding regions")
        
        # Show first few variants
        if len(coding_variants) > 0:
            display_cols = ['CHROM', 'POS', 'local_pos', 'REF', 'ALT', 'source_file']
            available_cols = [col for col in display_cols if col in coding_variants.columns]
            print("\nFirst few coding variants:")
            print(coding_variants[available_cols].head().to_string())
        
        # Create variant summary by position
        variant_positions = sorted(coding_variants['local_pos'].dropna().unique())
        
        if len(variant_positions) > 0:
            variant_info = []
            for local_pos in variant_positions:
                genomic_pos = genomic_to_local.get(local_pos, 0)
                pos_variants = coding_variants[coding_variants['local_pos'] == local_pos]
                
                ref_alleles = pos_variants['REF'].dropna().unique() if 'REF' in pos_variants.columns else []
                alt_alleles = pos_variants['ALT'].dropna().unique() if 'ALT' in pos_variants.columns else []
                source_files = pos_variants['source_file'].dropna().unique() if 'source_file' in pos_variants.columns else []
                
                variant_info.append({
                    'local_position': int(local_pos),
                    'genomic_position': int(genomic_pos),
                    'num_variants': len(pos_variants),
                    'ref_alleles': ','.join(map(str, ref_alleles)),
                    'alt_alleles': ','.join(map(str, alt_alleles)),
                    'source_files': ','.join(map(str, source_files))
                })
            
            variant_summary = pd.DataFrame(variant_info)
            print(f"\nVariant position summary: {len(variant_summary)} positions with variants")
            if len(variant_summary) > 0:
                print(variant_summary.head().to_string())
        else:
            print("No valid variant positions found")
    else:
        print("No variants found in coding regions")
elif len(combined_variants) > 0:
    print(f"Variants available ({len(combined_variants)}) but no gene coordinates for filtering")
else:
    print("No variant data to analyze")

## Step 5: Create Heatmap Visualization (Matplotlib Only)

In [ ]:
# Create heatmap visualization using only matplotlib
if coordinates_available:
    print("Creating heatmap visualization...")
    
    # Create figure with subplots
    fig = plt.figure(figsize=(18, 12))
    
    # Create custom subplot layout
    gs = fig.add_gridspec(3, 1, height_ratios=[1, 1.5, 4], hspace=0.3)
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])
    ax3 = fig.add_subplot(gs[2])
    
    # Panel 1: Gene structure overview
    ax1.set_xlim(roi_start, roi_end)
    ax1.set_ylim(-1, 2)
    
    # Draw gene body
    gene_rect = Rectangle((gene_start, 0), gene_end - gene_start, 0.5, 
                         facecolor='lightblue', edgecolor='blue', alpha=0.7)
    ax1.add_patch(gene_rect)
    
    # Draw CDS regions
    for start, end in cds_coords:
        cds_rect = Rectangle((start, 0), end - start, 0.5, 
                            facecolor='red', edgecolor='darkred', alpha=0.8)
        ax1.add_patch(cds_rect)
    
    # Mark variants if available
    if len(variant_summary) > 0:
        for _, variant in variant_summary.iterrows():
            ax1.axvline(x=variant['genomic_position'], color='orange', linewidth=2, alpha=0.8)
    
    ax1.set_title(f'cyp51A Gene (Afu4g06890) - Genomic Context\n{chromosome}:{roi_start:,}-{roi_end:,}', 
                  fontsize=14, fontweight='bold')
    ax1.set_ylabel('Gene Structure', fontsize=10)
    ax1.set_xticks([])
    ax1.grid(True, alpha=0.3)
    
    # Add legend for gene structure
    legend_elements = [
        plt.Rectangle((0, 0), 1, 1, facecolor='lightblue', alpha=0.7, label='Gene body'),
        plt.Rectangle((0, 0), 1, 1, facecolor='red', alpha=0.8, label='Coding sequence'),
        plt.Line2D([0], [0], color='orange', linewidth=2, label='Variant position')
    ]
    ax1.legend(handles=legend_elements, loc='upper right', fontsize=8)
    
    # Panel 2: Variant density
    if len(variant_summary) > 0:
        # Create variant density plot
        bar_width = max(50, (roi_end - roi_start) / 200)  # Adaptive bar width
        bars = ax2.bar(variant_summary['genomic_position'], variant_summary['num_variants'], 
                      width=bar_width, color='orange', alpha=0.7, edgecolor='darkorange')
        
        # Add value labels on bars
        for bar, count in zip(bars, variant_summary['num_variants']):
            height = bar.get_height()
            if height > 0:
                ax2.text(bar.get_x() + bar.get_width()/2., height + 0.1,
                        f'{int(count)}', ha='center', va='bottom', fontsize=8)
        
        ax2.set_ylabel('Variant Count', fontsize=10)
        ax2.set_title(f'Variant Density in cyp51A Gene ({len(variant_summary)} positions with variants)', fontsize=12)
        max_count = variant_summary['num_variants'].max()
        ax2.set_ylim(0, max_count * 1.2)
    else:
        ax2.text(0.5, 0.5, 'No variants found in coding regions', 
                ha='center', va='center', transform=ax2.transAxes, fontsize=14, 
                bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.7))
        ax2.set_ylim(0, 1)
    
    ax2.set_xlim(roi_start, roi_end)
    ax2.set_xticks([])
    ax2.grid(True, alpha=0.3)
    
    # Panel 3: Coding region nucleotide-level heatmap
    if total_coding_length > 0:
        print(f"Creating nucleotide-level heatmap for {total_coding_length} coding positions...")
        
        # Create data matrix
        matrix_rows = 6
        matrix = np.zeros((matrix_rows, total_coding_length))
        
        # Fill matrix with data
        for local_pos in range(1, total_coding_length + 1):
            col_idx = local_pos - 1
            genomic_pos = genomic_to_local.get(local_pos, 0)
            
            # Row 0: Local coordinate (normalized)
            matrix[0, col_idx] = local_pos / total_coding_length
            
            # Row 1: Codon position (1, 2, or 3)
            codon_pos = ((local_pos - 1) % 3) + 1
            matrix[1, col_idx] = codon_pos / 3.0  # Normalize to 0-1
            
            # Row 2: Distance from start (normalized)
            matrix[2, col_idx] = local_pos / total_coding_length
            
            # Row 3: Exon number (normalized)
            exon_num = 1
            if genomic_pos > 0:
                for i, (start, end) in enumerate(cds_coords, 1):
                    if start <= genomic_pos <= end:
                        exon_num = i
                        break
            matrix[3, col_idx] = exon_num / len(cds_coords) if len(cds_coords) > 0 else 0
            
            # Row 4: Variant presence
            has_variant = len(variant_summary) > 0 and local_pos in variant_summary['local_position'].values
            matrix[4, col_idx] = 1.0 if has_variant else 0.0
            
            # Row 5: Position within gene (normalized)
            if genomic_pos > 0:
                matrix[5, col_idx] = (genomic_pos - gene_start) / (gene_end - gene_start)
        
        # Create heatmap using imshow
        im = ax3.imshow(matrix, aspect='auto', cmap='viridis', interpolation='nearest')
        
        # Set row labels
        row_labels = ['Local Position', 'Codon Position', 'Distance from Start', 
                     'Exon Number', 'Variant Present', 'Gene Position']
        ax3.set_yticks(range(matrix_rows))
        ax3.set_yticklabels(row_labels, fontsize=10)
        
        # Set x-axis ticks (local coordinates)
        n_ticks = min(15, total_coding_length // 50)
        if n_ticks > 1:
            tick_positions = np.linspace(0, total_coding_length-1, n_ticks, dtype=int)
            tick_labels = [str(pos+1) for pos in tick_positions]
            ax3.set_xticks(tick_positions)
            ax3.set_xticklabels(tick_labels, rotation=45, fontsize=9)
        
        ax3.set_xlabel('Local Coding Coordinate (nucleotide position)', fontsize=11)
        ax3.set_title(f'Nucleotide-Level Analysis of cyp51A Coding Sequence ({total_coding_length} bp)', fontsize=12)
        
        # Add colorbar
        cbar = plt.colorbar(im, ax=ax3, shrink=0.6, aspect=30)
        cbar.set_label('Normalized Value\n(0=min, 1=max)', rotation=270, labelpad=20, fontsize=9)
        
        # Highlight variant positions
        if len(variant_summary) > 0:
            for _, variant in variant_summary.iterrows():
                local_pos = variant['local_position']
                if 1 <= local_pos <= total_coding_length:
                    ax3.axvline(x=local_pos-1, color='red', linewidth=1, alpha=0.8)
        
    else:
        ax3.text(0.5, 0.5, 'No coding sequence data available', 
                ha='center', va='center', transform=ax3.transAxes, fontsize=16,
                bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.7))
        ax3.set_xticks([])
        ax3.set_yticks([])
    
    # Save and show plot
    plt.tight_layout()
    plt.savefig('cyp51A_variants_heatmap.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\n" + "="*50)
    print("ANALYSIS SUMMARY")
    print("="*50)
    print(f"Gene: cyp51A (Afu4g06890)")
    print(f"Location: {chromosome}:{gene_start:,}-{gene_end:,}")
    print(f"Gene length: {gene_end - gene_start + 1:,} bp")
    print(f"Coding sequence: {total_coding_length} nucleotides")
    print(f"Number of exons: {len(cds_coords)}")
    print(f"VCF files processed: {len(vcf_data_loaded)}")
    print(f"Total variants found: {len(combined_variants)}")
    print(f"Variants in coding regions: {len(coding_variants)}")
    print(f"Unique positions with variants: {len(variant_summary)}")

else:
    print("Cannot create visualization - gene coordinate data not available")
    
    # Create simple info plot
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.text(0.5, 0.5, 
           'cyp51A Gene Analysis\n\n' + 
           'Status: Waiting for gene coordinate data\n' + 
           f'VCF files found: {len(vcf_files)}\n' + 
           f'Total variants loaded: {len(combined_variants)}\n\n' +
           'Please check GTF dataset #4 for gene annotation',
           ha='center', va='center', fontsize=14,
           bbox=dict(boxstyle='round,pad=1', facecolor='lightblue', alpha=0.7))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title('cyp51A Gene Analysis Status', fontsize=16, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('cyp51A_analysis_status.png', dpi=300, bbox_inches='tight')
    plt.show()

## Step 6: Export Results and Summary

In [ ]:
# Export analysis results
print("Exporting analysis results...")

# Create summary of what was accomplished
summary_data = {
    'analysis_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
    'gene_id': 'Afu4g06890',
    'gene_name': 'cyp51A',
    'gene_found': gene_found,
    'coordinates_available': coordinates_available,
    'total_coding_length': total_coding_length,
    'num_exons': len(cds_coords),
    'vcf_files_found': len(vcf_files),
    'vcf_files_loaded': len(vcf_data_loaded),
    'total_variants': len(combined_variants),
    'coding_variants': len(coding_variants),
    'variant_positions': len(variant_summary),
    'chromosome': chromosome if coordinates_available else 'N/A',
    'gene_start': gene_start if coordinates_available else 0,
    'gene_end': gene_end if coordinates_available else 0
}

# Save summary
summary_df = pd.DataFrame([summary_data])
summary_df.to_csv('cyp51A_analysis_summary.csv', index=False)
print("✓ Analysis summary saved")

# Export coordinate mapping if available
if coordinates_available and len(local_coords) > 0:
    coord_data = []
    for genomic_pos, local_pos in local_coords.items():
        exon_num = 1
        for i, (start, end) in enumerate(cds_coords, 1):
            if start <= genomic_pos <= end:
                exon_num = i
                break
        
        coord_data.append({
            'local_position': local_pos,
            'genomic_position': genomic_pos,
            'chromosome': chromosome,
            'codon_position': ((local_pos - 1) % 3) + 1,
            'exon_number': exon_num
        })
    
    coord_df = pd.DataFrame(coord_data).sort_values('local_position')
    coord_df.to_csv('cyp51A_coordinate_mapping.csv', index=False)
    print(f"✓ Coordinate mapping saved ({len(coord_df)} positions)")

# Export variant data if available
if len(variant_summary) > 0:
    variant_summary.to_csv('cyp51A_variant_summary.csv', index=False)
    print(f"✓ Variant summary saved ({len(variant_summary)} positions)")
    
    if len(coding_variants) > 0:
        # Select relevant columns for detailed export
        export_cols = ['CHROM', 'POS', 'local_pos', 'REF', 'ALT', 'source_file']
        available_export_cols = [col for col in export_cols if col in coding_variants.columns]
        
        coding_variants[available_export_cols].to_csv('cyp51A_coding_variants_detailed.csv', index=False)
        print(f"✓ Detailed variants saved ({len(coding_variants)} variant calls)")

# Export gene structure if available
if len(cds_coords) > 0:
    structure_data = []
    for i, (start, end) in enumerate(cds_coords, 1):
        # Find local coordinates for this exon
        exon_local_positions = [local_coords[pos] for pos in range(start, end + 1) if pos in local_coords]
        
        structure_data.append({
            'exon_number': i,
            'genomic_start': start,
            'genomic_end': end,
            'length_bp': end - start + 1,
            'local_start': min(exon_local_positions) if exon_local_positions else 0,
            'local_end': max(exon_local_positions) if exon_local_positions else 0
        })
    
    structure_df = pd.DataFrame(structure_data)
    structure_df.to_csv('cyp51A_gene_structure.csv', index=False)
    print(f"✓ Gene structure saved ({len(structure_df)} exons)")

# List all created files
output_files = [
    'cyp51A_analysis_summary.csv',
    'cyp51A_coordinate_mapping.csv', 
    'cyp51A_variant_summary.csv',
    'cyp51A_coding_variants_detailed.csv',
    'cyp51A_gene_structure.csv',
    'cyp51A_variants_heatmap.png',
    'cyp51A_analysis_status.png'
]

print("\n" + "="*40)
print("FILES CREATED:")
print("="*40)
for filename in output_files:
    if os.path.exists(filename):
        file_size = os.path.getsize(filename)
        print(f"✓ {filename} ({file_size} bytes)")
    else:
        print(f"- {filename} (not created)")
        
print("\n✓ Analysis complete!")